# LeetCode 539 — Minimum Time Difference bằng Python

Link đề: https://leetcode.com/problems/minimum-time-difference/ · Độ khó: **Medium** · Chủ đề: Array, Math, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findMinDifference(self, timePoints: List[str]) -> int:
        ...
```

Ghi chú: ví dụ cuối cùng trong mục 2 (có hai mốc giống hệt nhau và mốc "12:00") là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho danh sách các mốc giờ dạng `"HH:MM"` (đồng hồ 24 giờ), tìm **số phút nhỏ nhất** giữa **hai mốc bất kỳ** trong danh sách; chú ý đồng hồ là **vòng tròn**, `23:59` và `00:00` chỉ cách nhau **1 phút**.
2. Nắm các công cụ Python: cắt chuỗi `s[:2]`, `s[3:]`, `int()`, `divmod`, `sorted`, `zip(a, a[1:])`, `min` với generator, `set`, mảng/list boolean, **số nguyên lớn làm bitset** (`1 << m`, `&`, `|`, `>>`) và module `bisect`.
3. Đi từ **so sánh mọi cặp** → **sắp xếp rồi chỉ so các mốc kề nhau (cộng thêm cặp quấn vòng)** → **mảng 1440 ô (bucket)** → **bitset: xoay vòng rồi kiểm tra giao** → **hỏi "mốc kế tiếp theo vòng tròn" bằng tìm kiếm nhị phân**.
4. Hiểu mấu chốt: **đổi giờ sang phút** (`h * 60 + m`) để biến bài giờ giấc thành bài trên số nguyên; sau khi sắp xếp, cặp gần nhất luôn là **hai phần tử kề nhau** (cộng thêm cặp **cuối → đầu** đi vòng qua nửa đêm). Đồng thời dùng **nguyên lý chuồng bồ câu (pigeonhole)**: chỉ có `24 × 60 = 1440` mốc phân biệt, nên nếu có **hơn 1440** mốc thì chắc chắn có hai mốc trùng và đáp án là `0`.
5. Tránh các lỗi kinh điển: quên cặp quấn vòng, quên mốc trùng, đổi giờ bằng cách bỏ dấu `:` (`2359`), nhầm `+24` với `+1440`, và để brute force `O(n²)` chạy với `n = 2 × 10^4`.

## 2. Đề bài & ví dụ

Cho danh sách `timePoints` các mốc thời gian theo đồng hồ 24 giờ, mỗi mốc có định dạng **`"HH:MM"`**. Hãy trả về **hiệu số nhỏ nhất theo phút** giữa **hai mốc bất kỳ** trong danh sách (hai vị trí khác nhau trong danh sách).

Chú ý: đồng hồ quay vòng. Từ `23:59` đến `00:00` ngày hôm sau chỉ là `1` phút, không phải `1439` phút.

**Ví dụ 1**

- Input: `timePoints = ["23:59","00:00"]`
- Output: `1`
- Giải thích: đi từ `23:59` sang `00:00` qua nửa đêm mất 1 phút.

**Ví dụ 2**

- Input: `timePoints = ["00:00","23:59","00:00"]`
- Output: `0`
- Giải thích: hai mốc `"00:00"` giống hệt nhau nên chênh lệch `0`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `timePoints = ["00:00","12:00"]`
- Output: `720`
- Giải thích: hai mốc cách nhau đúng nửa vòng, `720` phút theo cả hai chiều; đây là đáp án **lớn nhất có thể**.

**Ràng buộc:**

- `2 <= timePoints.length <= 2 * 10^4`
- `timePoints[i]` có định dạng **`"HH:MM"`** (`00 <= HH <= 23`, `00 <= MM <= 59`).
- Các mốc **có thể trùng nhau**.

In [1]:
from typing import List
from itertools import pairwise
from bisect import bisect_right
from datetime import datetime
import random
import time

examples = [
    (["23:59", "00:00"], 1),
    (["00:00", "23:59", "00:00"], 0),
    (["00:00", "12:00"], 720),
]
for tp, expected in examples:
    print(tp, "-> kỳ vọng", expected)

['23:59', '00:00'] -> kỳ vọng 1
['00:00', '23:59', '00:00'] -> kỳ vọng 0
['00:00', '12:00'] -> kỳ vọng 720


## 3. Kiến thức nền

### 3.1. Cắt chuỗi và đổi sang số: `s[:2]`, `s[3:]`, `int`

Mốc `"13:14"` có đúng 5 ký tự: chỉ số `0,1` là giờ, chỉ số `2` là dấu `:`, chỉ số `3,4` là phút. Cắt lát (`slice`) `s[a:b]` lấy các ký tự từ chỉ số `a` tới **trước** `b`; bỏ `a` nghĩa là từ đầu, bỏ `b` nghĩa là tới hết. `int("07")` cho `7` (số `0` đứng đầu không sao).

In [2]:
s = "13:14"
print(s[:2], s[3:])                 # 13 14
print(int(s[:2]), int(s[3:]))       # 13 14
print(int("07"), int("00"))         # 7 0

# cách khác: tách theo dấu ':'
h, m = s.split(":")
print(h, m, int(h) * 60 + int(m))   # 13 14 794

13 14
13 14
7 0
13 14 794


### 3.2. Đổi giờ sang **phút kể từ 00:00**

Một ngày có `24 × 60 = 1440` phút. Mốc `HH:MM` ứng với `HH * 60 + MM` phút kể từ nửa đêm, nằm trong `[0, 1439]`. Việc đổi này biến thời gian thành **số nguyên** nên so sánh và trừ rất dễ: hiệu hai mốc chỉ là phép trừ.

Hàm `divmod(a, b)` cho cả thương và dư: `divmod(794, 60) = (13, 14)` — dùng để đổi ngược từ phút sang `HH:MM`.

In [3]:
def to_minutes(t):
    return int(t[:2]) * 60 + int(t[3:])

def to_clock(m):
    h, mm = divmod(m, 60)
    return f"{h:02d}:{mm:02d}"      # :02d = đủ 2 chữ số, thêm 0 ở đầu

print(to_minutes("00:00"), to_minutes("13:14"), to_minutes("23:59"))   # 0 794 1439
print(to_clock(794), to_clock(0), to_clock(1439))

0 794 1439
13:14 00:00 23:59


### 3.3. Đồng hồ là **vòng tròn**: khoảng cách theo hai chiều

Giữa hai mốc `a < b` (tính bằng phút) có hai đường đi quanh mặt đồng hồ: **thẳng** `b - a`, và **vòng qua nửa đêm** `1440 - (b - a)`. Khoảng cách thực là cái **nhỏ hơn**:

```
dist(a, b) = min(b - a, 1440 - (b - a))
```

Ví dụ `00:00` (0) và `23:59` (1439): `min(1439, 1) = 1`. Khoảng cách không bao giờ quá `720`.

In [4]:
def circ_dist(a, b):
    d = abs(a - b)
    return min(d, 1440 - d)

print(circ_dist(0, 1439))      # 1
print(circ_dist(0, 720))       # 720
print(circ_dist(100, 130))     # 30
print(circ_dist(100, 100))     # 0

1
720
30
0


### 3.4. `sorted`, `zip(a, a[1:])` và `itertools.pairwise`

`sorted(iterable)` trả về **list mới** đã sắp tăng dần (không đổi input; khác với `list.sort()` sắp **tại chỗ**). Để duyệt các **cặp kề nhau** `(a[i], a[i+1])` có hai cách tương đương: `zip(a, a[1:])` hoặc `itertools.pairwise(a)` (Python 3.10+).

In [5]:
a = sorted([794, 5, 1439, 60])
print(a)
print(list(zip(a, a[1:])))
print(list(pairwise(a)))
print("hiệu các cặp kề:", [y - x for x, y in zip(a, a[1:])])

[5, 60, 794, 1439]
[(5, 60), (60, 794), (794, 1439)]
[(5, 60), (60, 794), (794, 1439)]
hiệu các cặp kề: [55, 734, 645]


### 3.5. Nguyên lý chuồng bồ câu và `set`

Chỉ có `1440` mốc `HH:MM` khác nhau. Nếu danh sách có **hơn 1440** phần tử thì chắc chắn có hai phần tử **trùng** (chuồng bồ câu: nhét hơn 1440 con vào 1440 ô thì có ô chứa ≥ 2 con). Khi đó đáp án là `0` ngay, không cần tính gì thêm. Một `set` giúp kiểm tra trùng: `len(set(x)) < len(x)` nghĩa là có phần tử lặp.

In [6]:
tp = ["00:00", "23:59", "00:00"]
print(len(tp), len(set(tp)), len(set(tp)) < len(tp))        # có trùng
print("1440 mốc phân biệt:", len({to_clock(m) for m in range(1440)}))

3 2 True
1440 mốc phân biệt: 1440


### 3.6. Số nguyên lớn làm **bitset**

Python có số nguyên **không giới hạn độ lớn**, nên có thể dùng một số `B` làm dãy 1440 bit: bit thứ `m` bằng `1` nghĩa là "có mốc `m` phút". Các phép toán cần dùng:

- `1 << m`: số có đúng bit `m` bật.
- `B |= 1 << m`: bật bit `m`.
- `(B >> m) & 1`: đọc bit `m`.
- `B & C`: giao (bit nào bật ở cả hai).
- `B << d` dịch bit lên `d` vị trí; `B >> d` dịch xuống.

In [7]:
B = 0
for m in (0, 5, 1439):
    B |= 1 << m
print(bin(B)[-8:], "...", "bit 5 =", (B >> 5) & 1, "| bit 6 =", (B >> 6) & 1)
print("số bit bật:", bin(B).count("1"))
print("B & (B >> 5) có khác 0 không:", (B & (B >> 5)) != 0)      # bit 5 và bit 0 cách nhau 5

00100001 ... bit 5 = 1 | bit 6 = 0
số bit bật: 3
B & (B >> 5) có khác 0 không: True


### 3.7. `bisect_right`: tìm vị trí trong list đã sắp

`bisect_right(a, x)` trả về chỉ số `i` sao cho mọi phần tử `a[:i]` đều `<= x` và mọi phần tử `a[i:]` đều `> x` (với `a` đã sắp tăng). Nó chạy `O(log n)`. Nếu `i == len(a)` nghĩa là không có phần tử nào lớn hơn `x`.

In [8]:
u = [10, 20, 30, 40]
print(bisect_right(u, 25), bisect_right(u, 20), bisect_right(u, 40), bisect_right(u, 5))
# 2 (đứng trước 30), 2 (sau 20), 4 (hết list), 0

2 2 4 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `["23:59","00:00","12:30","12:31"]`. Đổi ra phút: `1439, 0, 750, 751`.

**Vẽ lên vòng tròn 1440 phút** và sắp theo vị trí tăng dần: `0, 750, 751, 1439`. Hai mốc gần nhau nhất chắc chắn phải **đứng cạnh nhau** trên vòng tròn (nếu giữa chúng còn một mốc khác thì cặp chứa mốc ở giữa còn gần hơn). Các cặp liền kề trên vòng gồm:

| cặp kề | hiệu |
|---|---|
| `0 → 750` | 750 |
| `750 → 751` | **1** |
| `751 → 1439` | 688 |
| `1439 → 0` (quấn vòng, tức `1439 → 1440`) | **1** |

Đáp án = **nhỏ nhất** trong các hiệu trên = `1`.

Mấu chốt: không cần so mọi cặp. Chỉ cần **sắp xếp** rồi xét **các cặp kề nhau** và **thêm một cặp quấn vòng** (mốc nhỏ nhất + `1440` trừ mốc lớn nhất). Cell dưới làm đúng việc đó bằng tay trên ví dụ này.

In [9]:
tp = ["23:59", "00:00", "12:30", "12:31"]
mins = sorted(to_minutes(t) for t in tp)
print("phút đã sắp:", mins)
gaps = [mins[i + 1] - mins[i] for i in range(len(mins) - 1)]
gaps.append(mins[0] + 1440 - mins[-1])          # cặp quấn vòng cuối -> đầu
print("các hiệu kề :", gaps)
print("đáp án      :", min(gaps))

phút đã sắp: [0, 750, 751, 1439]
các hiệu kề : [750, 1, 688, 1]
đáp án      : 1


## 5. Cách 1 — Brute Force / baseline: so sánh mọi cặp

**Ý tưởng.** Đổi mọi mốc ra phút. Với mỗi cặp `(i, j)`, `i < j`, tính khoảng cách vòng tròn `min(d, 1440 - d)` với `d = |a[i] - a[j]|`, lấy nhỏ nhất.

Cách này đúng nhưng **chậm**: `n(n-1)/2` cặp. Với `n = 2 × 10^4` là khoảng `2 × 10^8` cặp. Nhờ nguyên lý chuồng bồ câu ta có thể thêm dòng "nếu `n > 1440` thì trả `0`" để chặn trước, khi đó brute force còn tối đa `1440²/2 ≈ 10^6` cặp — nhưng ta tách rõ hai bản: **bản thô** (không dùng chuồng bồ câu) để thấy nút thắt, và nhớ rằng bản có chặn mới là bản dùng được.

In [10]:
def min_diff_brute(timePoints):
    nums = [to_minutes(t) for t in timePoints]
    best = 1440
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            d = abs(nums[i] - nums[j])
            best = min(best, d, 1440 - d)      # đi thẳng hoặc đi vòng qua nửa đêm
    return best


for tp, expected in examples:
    got = min_diff_brute(tp)
    print(tp, "->", got)
    assert got == expected

['23:59', '00:00'] -> 1
['00:00', '23:59', '00:00'] -> 0
['00:00', '12:00'] -> 720


**Độ phức tạp.** Thời gian **O(n²)**, bộ nhớ **O(n)** (list `nums`). Với `n = 2 × 10^4` tạo ra khoảng `2 × 10^8` phép tính — hàng chục giây trong Python, sẽ **TLE**.

**Nút thắt.** (1) Ta so cả những cặp rõ ràng không thể là cặp gần nhất. (2) Ta chưa tận dụng việc chỉ có `1440` giá trị khác nhau. Sắp xếp sẽ cho ta biết cặp tốt nhất nằm ở đâu.

## 6. Ý tưởng thuật toán / bước đột phá

Hai ý, ghép lại thành một lời giải gọn:

1. **Sắp xếp rồi chỉ so cặp kề nhau.** Sau khi đổi ra phút và sắp tăng dần, nếu hai mốc `x < z` không kề nhau thì tồn tại `y` ở giữa với `x ≤ y ≤ z`, và `y - x ≤ z - x`: cặp `(x, y)` không tệ hơn `(x, z)`. Vậy giá trị nhỏ nhất của `|a - b|` luôn đạt ở một cặp **kề nhau**.
2. **Quấn vòng qua nửa đêm.** Cặp (mốc lớn nhất, mốc nhỏ nhất) ở hai đầu cũng "kề nhau" trên vòng tròn: khoảng cách đi vòng là `nums[0] + 1440 - nums[-1]`. Mẹo gọn: **thêm `nums[0] + 1440` vào cuối list**, rồi chỉ cần tính hiệu các cặp kề.
3. **Chuồng bồ câu để chặn sớm**: `n > 1440` ⇒ trả `0`; nhờ đó mọi bước sau chỉ làm việc với tối đa `1440` mốc.

Vì chỉ có `1440` giá trị phút, ta còn có thể **bỏ hẳn bước sắp xếp** bằng mảng `1440` ô (cách 3), hoặc dùng bitset (cách 4), hoặc hỏi "mốc kế tiếp" bằng tìm kiếm nhị phân (cách 5).

## 7. Các cách tiếp theo

Tổng cộng 5 cách: cách 1 (brute force, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Sắp xếp + cặp kề + cặp quấn vòng

**Giải thích từng đoạn.**

- **Chặn sớm:** `if len(timePoints) > 1440: return 0`.
- **Đổi ra phút và sắp:** `nums = sorted(int(t[:2]) * 60 + int(t[3:]) for t in timePoints)`.
- **Thêm cặp quấn vòng:** `nums.append(nums[0] + 1440)`. Hiệu `nums[0] + 1440 - nums[-1]` chính là khoảng đi vòng từ mốc lớn nhất về mốc nhỏ nhất (qua nửa đêm).
- **Lấy min hiệu các cặp kề:** `min(b - a for a, b in pairwise(nums))`.

Nếu có hai mốc trùng, sau khi sắp chúng đứng kề nhau và hiệu là `0`, nên mốc trùng được xử lý tự nhiên.

In [11]:
def min_diff_sort(timePoints, verbose=False):
    if len(timePoints) > 1440:
        return 0
    nums = sorted(to_minutes(t) for t in timePoints)
    if verbose:
        print("sau khi sắp:", nums)
    nums.append(nums[0] + 1440)                   # cặp quấn vòng
    if verbose:
        print("thêm nums[0]+1440:", nums)
        print("hiệu các cặp kề :", [b - a for a, b in pairwise(nums)])
    return min(b - a for a, b in pairwise(nums))


for tp, expected in examples:
    got = min_diff_sort(tp, verbose=(tp == examples[0][0]))
    assert got == expected
print("OK")

sau khi sắp: [0, 1439]
thêm nums[0]+1440: [0, 1439, 1440]
hiệu các cặp kề : [1439, 1]
OK


**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    def findMinDifference(self, timePoints: List[str]) -> int:
        if len(timePoints) > 1440:
            return 0
        nums = sorted(int(t[:2]) * 60 + int(t[3:]) for t in timePoints)
        nums.append(nums[0] + 1440)
        return min(b - a for a, b in pairwise(nums))


for tp, expected in examples:
    assert Solution2().findMinDifference(tp) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Thời gian **O(n log n)** do sắp xếp (nhưng nhờ chặn `n > 1440`, thực tế `n ≤ 1440`, tức hằng số nhỏ); bộ nhớ **O(n)** cho list `nums`.

### 7.2. Cách 3 — Mảng 1440 ô (bucket / counting)

**Ý tưởng.** Vì chỉ có `1440` giá trị phút, ta dùng **mảng `seen` kích thước 1440** thay vì sắp xếp: `seen[m] = True` nếu đã có mốc `m`. Nếu gặp `seen[m]` đã `True` thì **trùng** → trả `0`. Sau đó **quét mảng từ `0` đến `1439`** theo thứ tự tăng dần: nhớ lại mốc `prev` trước đó, tính hiệu với mốc hiện tại, và nhớ mốc `first` (nhỏ nhất) và `last` (lớn nhất) để tính cặp quấn vòng `first + 1440 - last`.

Đây là **counting sort** thu nhỏ: thay vì so sánh, ta dùng chính chỉ số mảng làm khóa sắp.

In [13]:
def min_diff_bucket(timePoints):
    seen = [False] * 1440
    for t in timePoints:
        m = to_minutes(t)
        if seen[m]:
            return 0                           # trùng mốc -> chênh 0
        seen[m] = True

    best = 1440
    first = prev = None
    for m in range(1440):                      # quét theo thứ tự tăng dần
        if not seen[m]:
            continue
        if first is None:
            first = m
        else:
            best = min(best, m - prev)
        prev = m
    best = min(best, first + 1440 - prev)      # prev lúc này là mốc lớn nhất
    return best


for tp, expected in examples:
    got = min_diff_bucket(tp)
    print(tp, "->", got)
    assert got == expected

['23:59', '00:00'] -> 1
['00:00', '23:59', '00:00'] -> 0
['00:00', '12:00'] -> 720


**Code nộp LeetCode (cách 3):**

In [14]:
class Solution3:
    def findMinDifference(self, timePoints: List[str]) -> int:
        if len(timePoints) > 1440:
            return 0
        seen = [False] * 1440
        for t in timePoints:
            m = int(t[:2]) * 60 + int(t[3:])
            if seen[m]:
                return 0
            seen[m] = True
        best = 1440
        first = prev = -1
        for m in range(1440):
            if not seen[m]:
                continue
            if first < 0:
                first = m
            else:
                best = min(best, m - prev)
            prev = m
        return min(best, first + 1440 - prev)


for tp, expected in examples:
    assert Solution3().findMinDifference(tp) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Thời gian **O(n + 1440)** = **O(n + M)** với `M = 1440` hằng số: tuyến tính theo `n`, không có `log n`. Bộ nhớ **O(M) = O(1440)**, không phụ thuộc `n`. Đổi lại, nếu đề mở rộng sang giây (`86400`) hoặc mốc rất thưa, mảng sẽ lớn/lãng phí.

### 7.3. Cách 4 — Bitset và "xoay vòng": tìm `d` nhỏ nhất sao cho có cặp cách nhau `d`

**Ý tưởng.** Dùng một số nguyên `B` làm bitset: bit `m` bật nếu có mốc `m`. Nếu có trùng → `0`. Ngược lại, đáp án là **`d` nhỏ nhất** (`1 ≤ d ≤ 720`) sao cho **tồn tại hai mốc cách nhau `d` theo vòng tròn**.

Kiểm tra "tồn tại cặp cách `d`": **xoay vòng `B` đi `d` bit** (bit `m` chuyển sang bit `(m + d) mod 1440`) ra `rot(B, d)`, rồi tính `B & rot(B, d)`. Bit `y` của kết quả bật ⇔ `y ∈ B` và `y - d (mod 1440) ∈ B`, tức có hai mốc cách nhau `d` theo chiều kim đồng hồ. Cặp cách nhau `1440 - d` theo chiều đó cũng là cặp cách `d` theo chiều ngược lại, nên chỉ cần thử `d` đến `720`.

**Xoay vòng** `d` bit trên 1440 bit: `((B << d) | (B >> (1440 - d))) & MASK`, với `MASK = (1 << 1440) - 1` (cắt phần tràn ra ngoài 1440 bit).

In [15]:
def min_diff_bitset(timePoints, verbose=False):
    N = 1440
    MASK = (1 << N) - 1
    B = 0
    for t in timePoints:
        m = to_minutes(t)
        if (B >> m) & 1:
            return 0                                      # trùng
        B |= 1 << m

    def rotate(x, d):
        return ((x << d) | (x >> (N - d))) & MASK

    for d in range(1, N // 2 + 1):
        hit = B & rotate(B, d)
        if verbose:
            print(f"d = {d}: B & rot(B, d) {'!= 0 -> DỪNG' if hit else '== 0'}")
        if hit:
            return d
    return N // 2                                         # không tới được nếu n >= 2 và phân biệt


for tp, expected in examples:
    got = min_diff_bitset(tp)
    print(tp, "->", got)
    assert got == expected

['23:59', '00:00'] -> 1
['00:00', '23:59', '00:00'] -> 0
['00:00', '12:00'] -> 720


**Code nộp LeetCode (cách 4):**

In [16]:
class Solution4:
    def findMinDifference(self, timePoints: List[str]) -> int:
        if len(timePoints) > 1440:
            return 0
        N = 1440
        mask = (1 << N) - 1
        bits = 0
        for t in timePoints:
            m = int(t[:2]) * 60 + int(t[3:])
            if (bits >> m) & 1:
                return 0
            bits |= 1 << m
        for d in range(1, N // 2 + 1):
            if bits & (((bits << d) | (bits >> (N - d))) & mask):
                return d
        return N // 2


for tp, expected in examples:
    assert Solution4().findMinDifference(tp) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** Có tối đa `720` giá trị `d`, mỗi giá trị xử lý vài phép trên số `1440` bit (khoảng `1440/64 ≈ 23` từ máy) nên thời gian **O(n + M²/w)** với `M = 1440` và `w = 64` (độ rộng từ máy): cỡ vài chục nghìn thao tác từ, độc lập với `n` sau khi đã nạp. Bộ nhớ **O(M)** bit. Cách này không "tốt hơn" về mặt thực dụng nhưng cho một góc nhìn khác: **đáp án = khoảng cách nhỏ nhất có tự tương quan khác 0**.

### 7.4. Cách 5 — Với mỗi mốc, hỏi "mốc kế tiếp theo vòng tròn" bằng tìm kiếm nhị phân

**Ý tưởng.** Nếu có trùng → `0`. Ngược lại, lấy list `u` các phút **phân biệt đã sắp**. Với mỗi mốc `m`, **mốc kế tiếp theo chiều kim đồng hồ** là phần tử đầu tiên lớn hơn `m` (`u[bisect_right(u, m)]`), hoặc nếu không có thì quấn về `u[0]`. Khoảng cách tới nó là `(succ - m) mod 1440`. Đáp án là nhỏ nhất của các khoảng đó.

Khác cách 2 ở chỗ: ta không "ghép cặp kề" mà đặt **một câu hỏi truy vấn cho từng phần tử**, đây là khuôn mẫu hữu ích khi mốc được **thêm dần** (online) hoặc khi truy vấn "mốc gần `x` nhất".

In [17]:
def min_diff_bisect(timePoints, verbose=False):
    if len(timePoints) > 1440:
        return 0
    nums = [to_minutes(t) for t in timePoints]
    if len(set(nums)) < len(nums):
        return 0
    u = sorted(nums)
    best = 1440
    for m in nums:
        i = bisect_right(u, m)
        succ = u[i % len(u)]                    # i == len(u) -> quấn về u[0]
        gap = (succ - m) % 1440
        if verbose:
            print(f"m = {m:>4} -> i = {i}, succ = {succ:>4}, gap = {gap}")
        best = min(best, gap)
    return best


for tp, expected in examples:
    got = min_diff_bisect(tp, verbose=(tp == examples[0][0]))
    assert got == expected
print("OK")

m = 1439 -> i = 2, succ =    0, gap = 1
m =    0 -> i = 1, succ = 1439, gap = 1439
OK


**Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    def findMinDifference(self, timePoints: List[str]) -> int:
        if len(timePoints) > 1440:
            return 0
        nums = [int(t[:2]) * 60 + int(t[3:]) for t in timePoints]
        if len(set(nums)) < len(nums):
            return 0
        u = sorted(nums)
        best = 1440
        for m in nums:
            succ = u[bisect_right(u, m) % len(u)]
            best = min(best, (succ - m) % 1440)
        return best


for tp, expected in examples:
    assert Solution5().findMinDifference(tp) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp cách 5.** Sắp xếp `O(n log n)`, mỗi truy vấn `bisect` `O(log n)` nên tổng **O(n log n)**; bộ nhớ **O(n)**.

**Vì sao chỉ xét mốc kế tiếp theo một chiều?** Cặp gần nhất `(x, y)` luôn có dạng "`y` là mốc kế tiếp theo vòng tròn của `x`" (nếu giữa `x` và `y` theo chiều đó còn mốc khác thì cặp đó gần hơn). Vì vậy duyệt hết `x` và lấy `kế tiếp` là đủ.

## 8. TRACE từng bước

Ta trace trên `["23:59","00:00","12:30","12:31"]` (đáp án `1`) và `["00:00","23:59","12:00"]`.

### 8.1. Trace cách 2 (sắp xếp + cặp kề)

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `mốc` → `phút` | đổi `HH:MM` ra số phút |
| `cặp (a, b)` | hai phần tử kề nhau trong list đã sắp (kể cả phần tử cuối là `nums[0] + 1440`) |
| `b - a` | hiệu của cặp |
| `best` | nhỏ nhất đến lúc này |

In [19]:
def trace_sort(timePoints):
    print("n =", len(timePoints), "(<= 1440, không chặn sớm)")
    nums = []
    for t in timePoints:
        m = to_minutes(t)
        print(f"  {t} -> {t[:2]} * 60 + {t[3:]} = {m}")
        nums.append(m)
    nums.sort()
    print("sau khi sắp:", nums)
    nums.append(nums[0] + 1440)
    print("thêm nums[0] + 1440 =", nums[-1], "->", nums)
    best = 10**9
    for a, b in pairwise(nums):
        best = min(best, b - a)
        note = "  <- cặp quấn vòng (cuối -> đầu)" if b == nums[-1] else ""
        print(f"  cặp ({a:>4}, {b:>4}): b - a = {b - a:>4} | best = {best}{note}")
    return best

print("kết quả:", trace_sort(["23:59", "00:00", "12:30", "12:31"]))

n = 4 (<= 1440, không chặn sớm)
  23:59 -> 23 * 60 + 59 = 1439
  00:00 -> 00 * 60 + 00 = 0
  12:30 -> 12 * 60 + 30 = 750
  12:31 -> 12 * 60 + 31 = 751
sau khi sắp: [0, 750, 751, 1439]
thêm nums[0] + 1440 = 1440 -> [0, 750, 751, 1439, 1440]
  cặp (   0,  750): b - a =  750 | best = 750
  cặp ( 750,  751): b - a =    1 | best = 1
  cặp ( 751, 1439): b - a =  688 | best = 1
  cặp (1439, 1440): b - a =    1 | best = 1  <- cặp quấn vòng (cuối -> đầu)
kết quả: 1


Quan sát: hiệu nhỏ `1` xuất hiện **hai lần**: cặp `(750, 751)` (hai mốc `12:30`, `12:31`) và cặp quấn vòng `(1439, 1440)` — chính cặp mà người mới hay quên. Nếu bỏ cặp quấn vòng, ví dụ 1 `["23:59","00:00"]` sẽ ra `1439` thay vì `1`.

### 8.2. Trace cách 3 (mảng 1440 ô)

Ta không in 1440 dòng; chỉ in **các ô có mốc**, kèm `prev`, `first` và `best`.

In [20]:
def trace_bucket(timePoints):
    seen = [False] * 1440
    for t in timePoints:
        m = to_minutes(t)
        if seen[m]:
            print(f"gặp lại {t} (ô {m}) -> trùng, trả 0")
            return 0
        seen[m] = True
        print(f"đánh dấu ô {m} ({t})")
    best, first, prev = 1440, None, None
    for m in range(1440):
        if not seen[m]:
            continue
        if first is None:
            first = m
            print(f"ô {m:>4}: mốc nhỏ nhất, first = {first}")
        else:
            best = min(best, m - prev)
            print(f"ô {m:>4}: prev = {prev:>4}, m - prev = {m - prev:>4}, best = {best}")
        prev = m
    wrap = first + 1440 - prev
    best = min(best, wrap)
    print(f"cặp quấn vòng: first + 1440 - last = {first} + 1440 - {prev} = {wrap}, best = {best}")
    return best

print("kết quả:", trace_bucket(["00:00", "23:59", "12:00"]))

đánh dấu ô 0 (00:00)
đánh dấu ô 1439 (23:59)
đánh dấu ô 720 (12:00)
ô    0: mốc nhỏ nhất, first = 0
ô  720: prev =    0, m - prev =  720, best = 720
ô 1439: prev =  720, m - prev =  719, best = 719
cặp quấn vòng: first + 1440 - last = 0 + 1440 - 1439 = 1, best = 1
kết quả: 1


Với `["00:00","23:59","12:00"]` các ô có mốc là `0, 720, 1439`: hiệu kề là `720` và `719`, cặp quấn vòng là `0 + 1440 - 1439 = 1`, nên đáp án `1`.

### 8.3. Trace cách 4 (bitset xoay vòng)

Ta chọn ví dụ nhỏ `["00:00","00:05","00:12"]` (các bit `0, 5, 12`): cặp gần nhất là `(0, 5)` cách `5`, nên cần dừng ở `d = 5`. Ta in, với mỗi `d`, các **vị trí bit bật** của `B & rot(B, d)` (rỗng thì tiếp tục):

In [21]:
def trace_bitset(timePoints):
    N = 1440
    MASK = (1 << N) - 1
    B = 0
    for t in timePoints:
        B |= 1 << to_minutes(t)

    def positions(x):
        return [i for i in range(N) if (x >> i) & 1]

    print("bit bật của B:", positions(B))
    for d in range(1, N // 2 + 1):
        rot = ((B << d) | (B >> (N - d))) & MASK
        hit = B & rot
        print(f"d = {d}: rot(B, d) có bit {positions(rot)} | giao = {positions(hit)}")
        if hit:
            print("  -> giao khác rỗng, đáp án =", d)
            return d

print("kết quả:", trace_bitset(["00:00", "00:05", "00:12"]))

bit bật của B: [0, 5, 12]
d = 1: rot(B, d) có bit [1, 6, 13] | giao = []
d = 2: rot(B, d) có bit [2, 7, 14] | giao = []
d = 3: rot(B, d) có bit [3, 8, 15] | giao = []
d = 4: rot(B, d) có bit [4, 9, 16] | giao = []
d = 5: rot(B, d) có bit [5, 10, 17] | giao = [5]
  -> giao khác rỗng, đáp án = 5
kết quả: 5


Giải thích dòng `d = 5`: `rot(B, 5)` có các bit `5, 10, 17` (mỗi bit của `B` dịch thêm `5`). Giao với `B = {0, 5, 12}` là `{5}`, nghĩa là mốc `5` **và** mốc `5 - 5 = 0` cùng có mặt — hai mốc cách nhau đúng `5` phút.

### 8.4. Trace cách 5 (`bisect`)

Phần in `verbose=True` ở mục 7.4 đã cho thấy `i`, `succ`, `gap` cho từng mốc. Ở đây trace lại trên một ví dụ khác có quấn vòng rõ hơn:

In [22]:
print("kết quả:", min_diff_bisect(["23:50", "00:10", "12:00"], verbose=True))

m = 1430 -> i = 3, succ =   10, gap = 20
m =   10 -> i = 1, succ =  720, gap = 710
m =  720 -> i = 2, succ = 1430, gap = 710
kết quả: 20


Mốc `1430` (`23:50`) không có phần tử nào lớn hơn nên `i = len(u)`, `i % len(u) = 0` quấn về `u[0] = 10` và khoảng là `(10 - 1430) mod 1440 = 20` phút — đúng là từ `23:50` tới `00:10` hôm sau.

## 9. Code nộp LeetCode

Chọn **cách 2 (sắp xếp + cặp kề + cặp quấn vòng)**: ngắn, dễ nhớ, đúng và đủ nhanh. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta đã import ở trên):

In [23]:
class Solution:
    def findMinDifference(self, timePoints: List[str]) -> int:
        # Chuồng bồ câu: chỉ có 24 * 60 = 1440 mốc khác nhau
        if len(timePoints) > 1440:
            return 0
        # Đổi "HH:MM" -> số phút kể từ 00:00 rồi sắp tăng dần
        nums = sorted(int(t[:2]) * 60 + int(t[3:]) for t in timePoints)
        # Thêm mốc nhỏ nhất + 1440 để tính cả cặp quấn qua nửa đêm
        nums.append(nums[0] + 1440)
        # Nhỏ nhất của hiệu các cặp kề nhau
        return min(b - a for a, b in pairwise(nums))


for tp, expected in examples:
    got = Solution().findMinDifference(tp)
    print(tp, "->", got)
    assert got == expected

['23:59', '00:00'] -> 1
['00:00', '23:59', '00:00'] -> 0
['00:00', '12:00'] -> 720


## 10. Vì sao thuật toán đúng

**Bước 1 — Quy về số nguyên.** `HH:MM ↦ HH * 60 + MM ∈ [0, 1439]` là một song ánh giữa các mốc và số phút. Khoảng cách giữa hai mốc `a ≤ b` theo vòng tròn là `min(b - a, 1440 - (b - a))`.

**Bước 2 — Cặp tốt nhất là cặp kề (trên vòng tròn).** Sắp các mốc `p1 ≤ p2 ≤ ... ≤ pn`. Với hai mốc `pi ≤ pj` (`i < j`), có hai khoảng cách: đi thẳng `pj - pi` và đi vòng `1440 - (pj - pi)`.

- *Đi thẳng:* `pj - pi ≥ p(i+1) - pi` vì `p(i+1) ≤ pj`; nên cặp kề `(pi, p(i+1))` không tệ hơn.
- *Đi vòng:* `1440 - (pj - pi) = (pi + 1440) - pj`. Đặt `p(n+1) = p1 + 1440` (mốc `p1` ở "vòng sau"). Khi đó đi vòng bằng hiệu giữa `pi + 1440 ≥ p1 + 1440 = p(n+1)` và `pj ≤ pn`, nên `(pi + 1440) - pj ≥ p(n+1) - pn`; cặp quấn vòng `(pn, p(n+1))` không tệ hơn.

Vậy đáp án = `min` của các hiệu kề `p(k+1) - pk` với `k = 1..n` (trong đó `p(n+1) = p1 + 1440`) — chính là `min(b - a for a, b in pairwise(nums))` sau khi append.

**Mốc trùng.** Hai mốc trùng sau khi sắp nằm kề nhau nên hiệu là `0`, và `0` là giá trị nhỏ nhất có thể; ngoài ra nếu `n > 1440` thì theo chuồng bồ câu chắc chắn có trùng, trả `0` đúng.

**Giữ đúng dù `n = 2`.** Hai mốc `x ≤ y`: list sau append là `[x, y, x + 1440]`; hiệu `y - x` và `x + 1440 - y` — hai khoảng cách đi thẳng/đi vòng — lấy nhỏ nhất, đúng.

**Các cách khác.** Cách 3 là sắp xếp bằng chỉ số mảng (counting sort), phần còn lại giống cách 2. Cách 4 tìm `d` nhỏ nhất có cặp cách đúng `d` theo vòng; vì mọi cặp có khoảng cách vòng ≤ 720 nên duyệt `d` tăng dần dừng ở cặp tốt nhất. Cách 5 dùng quan sát: cặp tốt nhất luôn là (một mốc, mốc kế tiếp theo vòng của nó).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số mốc, `M = 1440` là số phút trong ngày.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | O(n²) | O(n) | `n ≤ 1440` sau chặn sớm thì ≈ 10⁶ cặp |
| 2. Sắp xếp + cặp kề | O(n log n) | O(n) | gọn, hay dùng nhất |
| 3. Mảng 1440 ô | O(n + M) | O(M) | không cần sắp xếp; khỏe khi `n` lớn |
| 4. Bitset + xoay vòng | O(n + M²/w) | O(M) bit | góc nhìn khác, ít dùng |
| 5. `bisect` tìm mốc kế tiếp | O(n log n) | O(n) | tiện cho dạng truy vấn/online |

**Nhờ chuồng bồ câu**, mọi cách ở trên chỉ chạy tối đa trên `n ≤ 1440` phần tử, nên ngay cả cách 1 và sắp xếp đều rất nhanh. Với `n = 2 × 10^4` mà **không** có dòng chặn sớm, brute force sẽ cần khoảng `2 × 10^8` cặp (TLE).

**Chi tiết cách 3.** `M = 1440` là hằng số nên coi như `O(n)`; nhưng nếu mở rộng đề sang giây (`86400`) thì `M` lớn và sắp xếp có thể tốt hơn khi `n` nhỏ.

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề cho cả 5 cách + bản nộp.
2. Edge case: `n = 2`, mốc trùng, đáp án lớn nhất `720`, cặp quấn vòng, toàn bộ 1440 mốc.
3. **Phản ví dụ** của các ý tưởng sai.
4. Đối chiếu với **oracle độc lập** dùng thư viện `datetime` (không dùng công thức `h*60+m` của ta) trên input ngẫu nhiên có seed.
5. Test lớn `n = 2 × 10^4`.

In [24]:
def oracle_datetime(timePoints):
    """Oracle độc lập: dùng datetime.strptime và timedelta, thử mọi cặp, xét hai chiều."""
    pts = [datetime.strptime(t, "%H:%M") for t in timePoints]
    best = None
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            diff = abs((pts[i] - pts[j]).total_seconds()) / 60
            circ = min(diff, 24 * 60 - diff)
            best = circ if best is None else min(best, circ)
    return int(best)


for tp, expected in examples:
    assert oracle_datetime(tp) == expected
print("oracle khớp ví dụ đề")

solvers = {
    "brute": min_diff_brute,
    "sort": min_diff_sort,
    "bucket": min_diff_bucket,
    "bitset": min_diff_bitset,
    "bisect": min_diff_bisect,
    "Solution2": lambda tp: Solution2().findMinDifference(tp),
    "Solution3": lambda tp: Solution3().findMinDifference(tp),
    "Solution4": lambda tp: Solution4().findMinDifference(tp),
    "Solution5": lambda tp: Solution5().findMinDifference(tp),
    "Solution": lambda tp: Solution().findMinDifference(tp),
}
for name, fn in solvers.items():
    for tp, expected in examples:
        assert fn(list(tp)) == expected, (name, tp)
print("tất cả cách khớp ví dụ đề:", list(solvers))

oracle khớp ví dụ đề
tất cả cách khớp ví dụ đề: ['brute', 'sort', 'bucket', 'bitset', 'bisect', 'Solution2', 'Solution3', 'Solution4', 'Solution5', 'Solution']


In [25]:
# --- Edge case ---
edge_cases = [
    ["00:00", "00:00"],                                  # n = 2, trùng
    ["00:00", "00:01"],                                  # n = 2, sát nhau
    ["00:00", "12:00"],                                  # lớn nhất có thể: 720
    ["06:00", "18:00"],                                  # cũng 720, không ở 00:00
    ["23:59", "00:00"],                                  # quấn vòng
    ["23:00", "01:00", "12:00"],                         # quấn vòng 120 phút
    ["01:00", "03:30", "06:00", "23:50"],                # đáp án 70 (23:50 -> 01:00)
    ["12:30", "12:31", "00:00", "23:59"],
    ["00:00", "23:59", "00:00"],
    ["13:14", "13:14", "05:05"],                         # trùng ở giữa
    ["00:00", "00:00", "00:00"],
]
for tp in edge_cases:
    want = oracle_datetime(tp)
    for name, fn in solvers.items():
        got = fn(list(tp))
        assert got == want, (name, tp, got, want)
    print(tp, "->", want)

# toàn bộ 1440 mốc (mọi phút trong ngày): đáp án 1
allm = [to_clock(m) for m in range(1440)]
for name, fn in solvers.items():
    if name == "brute":
        continue                                         # 1440^2/2 ~ 10^6 cặp, vẫn chạy được nhưng bỏ qua cho nhanh
    assert fn(list(allm)) == 1, name
print("1440 mốc phân biệt -> 1")

# 1441 mốc: chắc chắn có trùng -> 0
more = allm + ["10:10"]
for name in ("sort", "bucket", "bitset", "bisect", "Solution"):
    assert solvers[name](list(more)) == 0
print("1441 mốc -> 0 (chuồng bồ câu)")

['00:00', '00:00'] -> 0
['00:00', '00:01'] -> 1
['00:00', '12:00'] -> 720
['06:00', '18:00'] -> 720
['23:59', '00:00'] -> 1
['23:00', '01:00', '12:00'] -> 120
['01:00', '03:30', '06:00', '23:50'] -> 70
['12:30', '12:31', '00:00', '23:59'] -> 1
['00:00', '23:59', '00:00'] -> 0
['13:14', '13:14', '05:05'] -> 0
['00:00', '00:00', '00:00'] -> 0
1440 mốc phân biệt -> 1
1441 mốc -> 0 (chuồng bồ câu)


**Phản ví dụ.** Ba ý tưởng "nghe có vẻ đúng" nhưng sai:

- *Sai A:* sắp xếp rồi chỉ lấy min hiệu các cặp kề, **không** xét cặp quấn vòng.
- *Sai B:* đổi giờ bằng cách bỏ dấu `:` (`"23:59" -> 2359`) rồi trừ.
- *Sai C:* thêm `nums[0] + 24` (đơn vị giờ) thay vì `+ 1440` (đơn vị phút).

In [26]:
def wrong_no_wrap(tp):
    nums = sorted(to_minutes(t) for t in tp)
    return min(b - a for a, b in pairwise(nums))

def wrong_remove_colon(tp):
    nums = sorted(int(t.replace(":", "")) for t in tp)
    return min(b - a for a, b in pairwise(nums))

def wrong_plus_24(tp):
    nums = sorted(to_minutes(t) for t in tp)
    nums.append(nums[0] + 24)
    return min(b - a for a, b in pairwise(nums))

tp = ["23:59", "00:00"]
print("đúng          :", Solution().findMinDifference(tp))     # 1
print("sai A (no wrap):", wrong_no_wrap(tp))                    # 1439
print("sai B (bỏ ':') :", wrong_remove_colon(["00:59", "01:00"]), "(đúng là 1)")   # 41
print("sai C (+24)    :", wrong_plus_24(tp))                    # số âm

assert wrong_no_wrap(tp) != 1
assert wrong_remove_colon(["00:59", "01:00"]) != 1
assert wrong_plus_24(tp) < 0              # ra số âm vô nghĩa

đúng          : 1
sai A (no wrap): 1439
sai B (bỏ ':') : 41 (đúng là 1)
sai C (+24)    : -1415


Giải thích: **A** cho `1439` vì chỉ có cặp `(0, 1439)` với hiệu `1439`. **B**: `"00:59"` thành `59` và `"01:00"` thành `100` nên hiệu `41`, nhưng thực tế hai mốc cách nhau `1` phút (số `100` không phải là `60`). **C**: `0 + 24 - 1439` ra số âm nên `min` ra giá trị âm vô nghĩa — xem cell dưới.

In [27]:
nums = sorted(to_minutes(t) for t in ["23:59", "00:00"])
nums.append(nums[0] + 24)
print(nums, [b - a for a, b in pairwise(nums)])      # [0, 1439, 24] -> [1439, -1415]

[0, 1439, 24] [1439, -1415]


**Đối chiếu nhiều input ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh `n` từ `2` đến `30` mốc: lúc thì lấy trong **toàn bộ 1440 phút** (ít trùng), lúc thì lấy trong **một khoảng hẹp** (hay trùng), lúc thì lấy quanh **nửa đêm** (hay phải quấn vòng).

In [28]:
def random_times(rng, n, mode):
    if mode == "wide":
        pool = range(1440)
    elif mode == "narrow":
        start = rng.randrange(1440)
        pool = [(start + k) % 1440 for k in range(rng.randint(2, 40))]
    else:                                    # quanh nửa đêm
        pool = [(1440 - 15 + k) % 1440 for k in range(30)]
    return [to_clock(rng.choice(pool)) for _ in range(n)]


rng = random.Random(539)
cnt = 0
for _ in range(600):
    n = rng.randint(2, 30)
    mode = rng.choice(["wide", "narrow", "midnight"])
    tp = random_times(rng, n, mode)
    want = oracle_datetime(tp)
    for name, fn in solvers.items():
        original = list(tp)
        got = fn(tp)
        assert got == want, (name, tp, got, want)
        assert tp == original, (name, "đã làm thay đổi input")
    cnt += 1
print("đối chiếu", cnt, "input ngẫu nhiên: tất cả cách khớp oracle datetime và không sửa input")

đối chiếu 600 input ngẫu nhiên: tất cả cách khớp oracle datetime và không sửa input


In [29]:
# Test lớn: n = 2 * 10^4 (chắc chắn trùng -> 0) và n = 1440 phân biệt thưa/dày
rng = random.Random(5390)
big = [to_clock(rng.randrange(1440)) for _ in range(2 * 10**4)]
for name in ("sort", "bucket", "bitset", "bisect", "Solution2", "Solution3", "Solution4", "Solution5", "Solution"):
    assert solvers[name](list(big)) == 0
print("n = 2*10^4 ngẫu nhiên -> 0")

# 1000 mốc phân biệt ngẫu nhiên: so các cách nhanh với nhau và với oracle sắp xếp độc lập
pool = rng.sample(range(1440), 1000)
tp = [to_clock(m) for m in pool]
ps = sorted(pool)
indep = min(min(b - a for a, b in zip(ps, ps[1:])), ps[0] + 1440 - ps[-1])
for name in ("sort", "bucket", "bitset", "bisect", "Solution2", "Solution3", "Solution4", "Solution5", "Solution"):
    assert solvers[name](list(tp)) == indep, name
print("1000 mốc phân biệt ->", indep)

# thưa: 2 mốc cách xa -> gần 720
for _ in range(50):
    a = rng.randrange(1440)
    b = (a + rng.randint(1, 1439)) % 1440
    tp = [to_clock(a), to_clock(b)]
    d = abs(a - b)
    want = min(d, 1440 - d)
    for name in ("sort", "bucket", "bitset", "bisect", "Solution"):
        assert solvers[name](list(tp)) == want
print("50 cặp ngẫu nhiên n = 2: OK")

n = 2*10^4 ngẫu nhiên -> 0
1000 mốc phân biệt -> 1
50 cặp ngẫu nhiên n = 2: OK


In [30]:
# So sánh thời gian: n = 1440 phân biệt (đủ lớn để brute force bắt đầu chậm)
tp = [to_clock(m) for m in random.Random(1).sample(range(1440), 1440)]
for name in ("brute", "sort", "bucket", "bisect", "bitset"):
    t0 = time.perf_counter()
    r = solvers[name](list(tp))
    dt = time.perf_counter() - t0
    print(f"{name:>7}: {dt * 1000:9.2f} ms -> {r}")

  brute:    124.57 ms -> 1
   sort:      0.89 ms -> 1
 bucket:      0.43 ms -> 1
 bisect:      1.00 ms -> 1
 bitset:      0.58 ms -> 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên cặp **quấn vòng** qua nửa đêm

Chỉ xét hiệu các cặp kề trong list đã sắp mà không thêm `nums[0] + 1440`.

In [31]:
def wrong_missing_wrap(timePoints):
    nums = sorted(to_minutes(t) for t in timePoints)
    return min(b - a for a, b in zip(nums, nums[1:]))

tp = ["23:59", "00:00"]
print("sai  :", wrong_missing_wrap(tp))                   # 1439
print("đúng :", Solution().findMinDifference(tp))         # 1

sai  : 1439
đúng : 1


**Vì sao sai.** Sau khi sắp, hai mốc là `0` và `1439`, hiệu kề `1439`; nhưng đi vòng qua nửa đêm chỉ `1` phút. **Cách sửa:** `nums.append(nums[0] + 1440)` trước khi tính hiệu các cặp kề.

### Lỗi 2: đổi giờ bằng cách bỏ dấu `:`

`int("0100")` là `100`, nhưng `01:00` là `60` phút. Số "thập phân ghép" `HHMM` **không** tuyến tính theo thời gian (nhảy từ `59` sang `100`).

In [32]:
def wrong_parse(timePoints):
    nums = sorted(int(t.replace(":", "")) for t in timePoints)
    nums.append(nums[0] + 1440)
    return min(b - a for a, b in zip(nums, nums[1:]))

tp = ["00:59", "01:00"]
print("sai  :", wrong_parse(tp), "(đúng là", Solution().findMinDifference(tp), ")")

sai  : 41 (đúng là 1 )


**Vì sao sai.** `"00:59" → 59`, `"01:00" → 100`, hiệu `41` thay vì `1` phút. **Cách sửa:** luôn đổi `h * 60 + m`.

### Lỗi 3: nhầm đơn vị khi quấn vòng (`+ 24` thay cho `+ 1440`)

Ta đang tính bằng **phút** nên một vòng là `1440`, không phải `24` (giờ) hay `60`.

In [33]:
def wrong_unit(timePoints):
    nums = sorted(to_minutes(t) for t in timePoints)
    nums.append(nums[0] + 24)           # SAI đơn vị
    return min(b - a for a, b in zip(nums, nums[1:]))

tp = ["23:59", "00:00"]
print("sai  :", wrong_unit(tp), "(ra số âm vô nghĩa)")
print("đúng :", Solution().findMinDifference(tp))

sai  : -1415 (ra số âm vô nghĩa)
đúng : 1


**Vì sao sai.** `0 + 24 - 1439 = -1415`. Khoảng cách không thể âm. **Cách sửa:** `nums[0] + 1440`.

### Lỗi 4: dùng mảng bucket nhưng **mất thông tin trùng**

Với mảng boolean, đặt `seen[m] = True` hai lần không để lại dấu vết. Nếu không kiểm tra trùng **trước khi đánh dấu**, hai mốc giống nhau bị coi như một và đáp án sai (đúng ra là `0`).

In [34]:
def wrong_bucket_no_dup(timePoints):
    seen = [False] * 1440
    for t in timePoints:
        seen[to_minutes(t)] = True              # KHÔNG kiểm tra trùng
    pts = [m for m in range(1440) if seen[m]]
    pts.append(pts[0] + 1440)
    return min(b - a for a, b in zip(pts, pts[1:]))

tp = ["00:00", "00:00", "05:00"]
print("sai  :", wrong_bucket_no_dup(tp), "(đúng là", Solution().findMinDifference(tp), ")")

sai  : 300 (đúng là 0 )


**Vì sao sai.** Sau khi gộp, chỉ còn `{0, 300}` nên hiệu nhỏ nhất `300`, nhưng đề có hai mốc `"00:00"` nên đáp án là `0`. **Cách sửa:** `if seen[m]: return 0` ngay khi gặp ô đã đánh dấu (như `min_diff_bucket`).

### Lỗi 5: không chặn sớm với `n` lớn → brute force TLE

Brute force `O(n²)` với `n = 2 × 10^4` là khoảng `2 × 10^8` cặp. Hãy đo thử với `n = 3000` (đã đủ thấy sự khác biệt):

In [35]:
rng = random.Random(13)
tp = [to_clock(rng.randrange(1440)) for _ in range(3000)]
t0 = time.perf_counter()
r1 = min_diff_brute(tp)                 # không chặn sớm
t1 = time.perf_counter()
r2 = Solution().findMinDifference(tp)   # chặn sớm n > 1440
t2 = time.perf_counter()
print(f"brute không chặn: {(t1 - t0) * 1000:8.1f} ms -> {r1}")
print(f"có chặn sớm     : {(t2 - t1) * 1000:8.3f} ms -> {r2}")
assert r1 == r2 == 0

brute không chặn:    558.4 ms -> 0
có chặn sớm     :    0.054 ms -> 0


**Vì sao chậm.** Brute force không biết rằng `n > 1440` thì chắc chắn có trùng; nó vẫn duyệt hết `n²/2` cặp. **Cách sửa:** thêm dòng `if len(timePoints) > 1440: return 0` ở đầu hàm — đây cũng là lý do đề cho `n` tới `2 × 10^4`.

### Lỗi 6: dùng `list.sort()` làm thay đổi input / quên `sorted` trả về list mới

`timePoints.sort()` sắp **tại chỗ** và trả về `None`; viết `nums = nums.sort()` làm `nums` thành `None`.

In [36]:
nums = [3, 1, 2]
res = nums.sort()
print("nums.sort() trả về:", res, "| nums =", nums)       # None [1, 2, 3]

try:
    bad = [3, 1, 2]
    bad = bad.sort()
    print(len(bad))
except TypeError as e:
    print("TypeError:", e)
print("sorted trả về list mới:", sorted([3, 1, 2]))

nums.sort() trả về: None | nums = [1, 2, 3]
TypeError: object of type 'NoneType' has no len()
sorted trả về list mới: [1, 2, 3]


**Cách sửa.** Dùng `nums = sorted(...)` hoặc gọi `nums.sort()` mà **không** gán lại kết quả. Với bài này `sorted(generator)` đúng ý vì ta đang tạo list phút từ list chuỗi.

## 14. Mẹo & nhận dạng pattern

- **"Khoảng cách nhỏ nhất giữa hai phần tử"** → **sắp xếp** rồi xét **cặp kề**. Đây là pattern kinh điển (xem cả "Minimum Absolute Difference").
- **Đại lượng có tính quay vòng (giờ, góc, chỉ số trong mảng tròn)** → xét thêm **cặp quấn vòng** bằng cách nối `nums[0] + chu_kỳ` vào cuối; hoặc dùng phép `% chu_kỳ`.
- **Miền giá trị nhỏ và cố định (như `1440` phút)** → nghĩ ngay tới **mảng đếm / bucket / counting sort** và **chuồng bồ câu** để chặn sớm.
- **Đổi định dạng khó thành số nguyên**: `HH:MM → HH*60 + MM`; mọi phép so sánh/trừ sau đó đều dễ. Đừng ghép chữ số.
- **Nhìn ràng buộc mà suy ra mẹo**: `n ≤ 2 × 10^4` nhưng chỉ `1440` giá trị khác nhau — đề đang gợi ý chuồng bồ câu.
- **`pairwise(nums)`** (hoặc `zip(nums, nums[1:])`) rất tiện cho "mọi cặp kề".
- **Đừng quên input có thể trùng nhau**; hãy tự hỏi "nếu hai phần tử bằng nhau thì sao?" cho mọi cách giải.

## 15. Biến thể

### 15.1. Mốc giờ tới **giây** (`"HH:MM:SS"`, 86400 giây)

Chỉ đổi hằng số: `h * 3600 + m * 60 + s` và chu kỳ `86400`. Chặn sớm khi `n > 86400`.

In [37]:
def min_diff_seconds(times):
    if len(times) > 86400:
        return 0
    nums = sorted(int(t[:2]) * 3600 + int(t[3:5]) * 60 + int(t[6:]) for t in times)
    nums.append(nums[0] + 86400)
    return min(b - a for a, b in pairwise(nums))

print(min_diff_seconds(["23:59:59", "00:00:00"]))         # 1
print(min_diff_seconds(["00:00:00", "00:00:30", "12:00:00"]))   # 30
assert min_diff_seconds(["23:59:59", "00:00:00"]) == 1
assert min_diff_seconds(["00:00:00", "00:00:30", "12:00:00"]) == 30

1
30


### 15.2. Trả về **cặp** mốc gần nhất (không chỉ khoảng cách)

Giữ thêm chỉ số/mốc tạo ra hiệu nhỏ nhất khi duyệt các cặp kề.

In [38]:
def closest_pair(timePoints):
    pts = sorted(set(timePoints), key=to_minutes)
    if len(pts) < len(timePoints):
        dup = next(t for t in pts if timePoints.count(t) > 1)
        return (dup, dup, 0)
    best = None
    for i in range(len(pts)):
        a, b = pts[i], pts[(i + 1) % len(pts)]
        d = (to_minutes(b) - to_minutes(a)) % 1440
        if best is None or d < best[2]:
            best = (a, b, d)
    return best

print(closest_pair(["23:59", "00:00", "12:30"]))
assert closest_pair(["23:59", "00:00", "12:30"]) == ("23:59", "00:00", 1)
assert closest_pair(["10:00", "10:00", "05:00"]) == ("10:00", "10:00", 0)

('23:59', '00:00', 1)


### 15.3. **Khoảng trống lớn nhất** (lâu nhất không có mốc nào)

Cùng khung "sắp xếp + cặp kề + quấn vòng" nhưng lấy `max` thay vì `min`.

In [39]:
def max_gap(timePoints):
    nums = sorted(to_minutes(t) for t in timePoints)
    nums.append(nums[0] + 1440)
    return max(b - a for a, b in pairwise(nums))

print(max_gap(["00:00", "12:00"]))                 # 720
print(max_gap(["23:59", "00:00", "12:30"]))        # 750 (từ 00:00 đến 12:30)
assert max_gap(["00:00", "12:00"]) == 720
assert max_gap(["23:59", "00:00", "12:30"]) == 750

720
750


### 15.4. Định dạng **12 giờ** `"hh:mm AM/PM"`

Chỉ thay phần đổi ra phút (nhớ `12 AM` là `0` giờ và `12 PM` là `12` giờ).

In [40]:
def to_minutes_12h(t):
    hm, ap = t.split(" ")
    h, m = map(int, hm.split(":"))
    h %= 12                       # 12 -> 0
    if ap == "PM":
        h += 12
    return h * 60 + m

print(to_minutes_12h("12:00 AM"), to_minutes_12h("12:00 PM"), to_minutes_12h("11:59 PM"))   # 0 720 1439
assert [to_minutes_12h(x) for x in ("12:00 AM", "12:00 PM", "11:59 PM")] == [0, 720, 1439]

0 720 1439


### 15.5. Biến thể để tự nghĩ

- **Minimum Absolute Difference (1200)**: không có quấn vòng, chỉ sắp xếp + cặp kề.
- **Meeting Rooms (252)** / **Merge Intervals (56)**: cũng sắp xếp theo mốc thời gian rồi xét kề nhau.
- **Thêm mốc dần dần** (online): giữ cấu trúc cây/`SortedList` để hỏi "mốc kế tiếp/kề trước" — liên hệ cách 5.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | so mọi cặp, lấy `min(d, 1440 - d)` | O(n²) | O(n) | chỉ làm oracle/hiểu đề; cần chặn `n > 1440` |
| 2. Sắp xếp + cặp kề | sắp, thêm `nums[0] + 1440`, min hiệu kề | O(n log n) | O(n) | cách nộp bài mặc định, ngắn và an toàn |
| 3. Mảng 1440 ô | counting sort theo chỉ số phút | O(n + M) | O(M) | miền giá trị nhỏ cố định, muốn tránh `log n` |
| 4. Bitset xoay vòng | tìm `d` nhỏ nhất sao cho `B & rot(B, d) ≠ 0` | O(n + M²/w) | O(M) bit | để học tư duy bit; ít khi dùng thực tế |
| 5. `bisect` mốc kế tiếp | mỗi mốc hỏi successor theo vòng | O(n log n) | O(n) | dạng truy vấn/online, mở rộng "mốc gần `x` nhất" |

**Lời khuyên.** Học cách 2 trước (cốt lõi) và nhớ dòng chặn `n > 1440`. Cách 3 là tối ưu thực dụng khi miền giá trị nhỏ; cách 1 giúp kiểm chứng; cách 4, 5 mở rộng tư duy.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [41]:
print(Solution().findMinDifference(["01:00", "02:30", "05:00"]))
print(Solution().findMinDifference(["23:30", "00:15"]))
print(Solution().findMinDifference(["10:10", "10:10"]))

90
45
0


**Bài 2 (tính tay).** Với `["22:00", "01:00", "13:00", "06:30"]`, đổi ra phút, sắp tăng dần, liệt kê các hiệu kề **kể cả cặp quấn vòng**, rồi gán đáp án vào `ans2` (đang là `None`).

In [42]:
ans2 = None      # TODO: một số nguyên (số phút)

**Bài 3 (điền TODO).** Viết `circular_dist(t1, t2)` nhận **hai chuỗi `"HH:MM"`** và trả về khoảng cách vòng tròn theo phút. Ví dụ `("23:30", "00:15") -> 45`.

In [43]:
def circular_dist(t1, t2):
    # TODO: đổi ra phút, rồi min(d, 1440 - d)
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_min_diff(timePoints)` theo cách **mảng 1440 ô** (cách 3), nhớ xử lý mốc trùng và cặp quấn vòng.

In [44]:
def my_min_diff(timePoints):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `nearest_time(timePoints, query)` trả về **mốc trong danh sách gần `query` nhất** (khoảng cách vòng tròn); nếu có hai mốc cùng khoảng cách thì trả về mốc có **số phút nhỏ hơn**. (b) Viết `can_all_be_k_apart(timePoints, k)` trả về `True` nếu **mọi cặp** mốc cách nhau **ít nhất `k` phút** (theo vòng tròn). Gợi ý cho (b): bài này chính là so sánh `k` với đáp án của bài gốc.

In [45]:
def nearest_time(timePoints, query):
    # TODO
    pass

def can_all_be_k_apart(timePoints, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: đổi từng mốc ra phút rồi tính hiệu kề; mốc trùng cho `0`.
- Bài 2: `22:00 = 1320`, `01:00 = 60`, ...
- Bài 3: dùng hàm `to_minutes` ở mục 3.2.
- Bài 4: mảng `seen = [False] * 1440`.
- Bài 5: (a) thử khoảng cách vòng tới từng mốc; (b) lấy đáp án gốc rồi so với `k`.

**Mức 2 (vừa):**
- Bài 2: các phút sau sắp: `60, 390, 780, 1320`; cặp quấn vòng `60 + 1440 - 1320 = 180`.
- Bài 3: `d = abs(a - b)` rồi `min(d, 1440 - d)`.
- Bài 4: quét `m` từ `0` đến `1439`, nhớ `first` và `prev`; cuối cùng `first + 1440 - prev`.
- Bài 5: (a) `min(timePoints, key=lambda t: (circular_dist(t, query), to_minutes(t)))`; (b) nếu có trùng thì khoảng cách `0 < k` (với `k ≥ 1`) → `False`; nếu `k <= 0` luôn `True`.

**Mức 3 (gần đáp án):**
- Bài 3: xem `circ_dist` ở mục 3.3.
- Bài 4: xem cách 3 ở mục 7.2, bỏ phần in.
- Bài 5: (b) `return Solution().findMinDifference(timePoints) >= k` (cần `len(timePoints) >= 2`).

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [46]:
# Bài 1: 90 phút giữa 01:00 và 02:30; 23:30 -> 00:15 là 45; trùng là 0
assert Solution().findMinDifference(["01:00", "02:30", "05:00"]) == 90
assert Solution().findMinDifference(["23:30", "00:15"]) == 45
assert Solution().findMinDifference(["10:10", "10:10"]) == 0

# Bài 2: phút sau sắp: 60, 390, 780, 1320; hiệu kề: 330, 390, 540; quấn vòng: 60 + 1440 - 1320 = 180
ans2 = 180
assert ans2 == Solution().findMinDifference(["22:00", "01:00", "13:00", "06:30"])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [47]:
# Bài 3
def circular_dist(t1, t2):
    d = abs(to_minutes(t1) - to_minutes(t2))
    return min(d, 1440 - d)

assert circular_dist("23:30", "00:15") == 45
assert circular_dist("00:00", "12:00") == 720
assert circular_dist("05:05", "05:05") == 0
assert circular_dist("00:59", "01:00") == 1
print("Bài 3 OK")

Bài 3 OK


In [48]:
# Bài 4
def my_min_diff(timePoints):
    seen = [False] * 1440
    for t in timePoints:
        m = to_minutes(t)
        if seen[m]:
            return 0
        seen[m] = True
    best, first, prev = 1440, None, None
    for m in range(1440):
        if seen[m]:
            if first is None:
                first = m
            else:
                best = min(best, m - prev)
            prev = m
    return min(best, first + 1440 - prev)

rng = random.Random(4)
for _ in range(300):
    tp = random_times(rng, rng.randint(2, 25), rng.choice(["wide", "narrow", "midnight"]))
    assert my_min_diff(tp) == oracle_datetime(tp), tp
assert my_min_diff(["23:59", "00:00"]) == 1
print("Bài 4 OK")

Bài 4 OK


In [49]:
# Bài 5
def nearest_time(timePoints, query):
    return min(timePoints, key=lambda t: (circular_dist(t, query), to_minutes(t)))

def can_all_be_k_apart(timePoints, k):
    if k <= 0:
        return True
    return Solution().findMinDifference(timePoints) >= k

assert nearest_time(["01:00", "23:00", "12:00"], "00:00") in ("01:00", "23:00")
assert nearest_time(["01:00", "23:00", "12:00"], "00:00") == "01:00"        # hòa 60/60 -> phút nhỏ hơn (01:00)
assert nearest_time(["23:59", "12:00"], "00:01") == "23:59"
assert nearest_time(["05:00"], "17:30") == "05:00"
assert can_all_be_k_apart(["00:00", "00:30", "01:00"], 30) is True
assert can_all_be_k_apart(["00:00", "00:30", "01:00"], 31) is False
assert can_all_be_k_apart(["23:59", "00:01"], 3) is False
assert can_all_be_k_apart(["10:00", "10:00"], 1) is False
rng = random.Random(5)
for _ in range(200):
    tp = random_times(rng, rng.randint(2, 20), "wide")
    k = rng.randint(1, 60)
    want = oracle_datetime(tp) >= k
    assert can_all_be_k_apart(tp, k) == want
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (sắp xếp + cặp kề + quấn vòng):**

```python
from itertools import pairwise

class Solution:
    def findMinDifference(self, timePoints: List[str]) -> int:
        if len(timePoints) > 1440:          # chuồng bồ câu -> chắc chắn có trùng
            return 0
        nums = sorted(int(t[:2]) * 60 + int(t[3:]) for t in timePoints)
        nums.append(nums[0] + 1440)          # cặp quấn vòng qua nửa đêm
        return min(b - a for a, b in pairwise(nums))
```

**Template (mảng 1440 ô):**

```python
seen = [False] * 1440
for t in timePoints:
    m = int(t[:2]) * 60 + int(t[3:])
    if seen[m]:
        return 0
    seen[m] = True
best, first, prev = 1440, -1, -1
for m in range(1440):
    if seen[m]:
        if first < 0:
            first = m
        else:
            best = min(best, m - prev)
        prev = m
return min(best, first + 1440 - prev)
```

**Câu thần chú:** *đổi sang phút → sắp xếp → hiệu các cặp kề → nhớ cặp quấn vòng (+1440) → n > 1440 thì 0.*

**Checklist trước khi submit:**

- [ ] Đã đổi `HH:MM` sang **phút** (`h * 60 + m`), không ghép chữ số?
- [ ] Có cặp **quấn vòng** (`nums[0] + 1440`, đúng đơn vị phút)?
- [ ] Mốc **trùng** cho `0` (và chặn sớm `n > 1440`)?
- [ ] Dùng `sorted(...)` (hoặc không gán lại kết quả của `list.sort()`)?
- [ ] Không sửa input ngoài ý muốn?
- [ ] Đã thử: `["23:59","00:00"]`, hai mốc trùng, `["00:00","12:00"]` (đáp án 720), `n = 2`, `n > 1440`?